# UMTAM Multi-Task Merging: 7 GLUE Tasks

**Section 4.3: Multi-Task Merging Results**

This notebook validates UMTAM's curvature-aware model merging (Algorithm 2) by:
1. Training 7 task-specific BERT experts
2. Merging them using multiple methods
3. Evaluating merged models on all tasks

Merging methods compared:
- **UMTAM** (curvature-aware with sign election)
- **TIES-Merging** (magnitude-based trim + elect + merge)
- **DARE + TIES** (random drop + TIES)
- **Linear Averaging** (simple task arithmetic)
- **Magnitude-Weighted** (weight by parameter magnitude)

**Runtime estimate**: ~3-4 hours on Colab GPU (T4/L4)

## Setup & Installation

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import warnings
import copy
import gc
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Task Configuration

In [ ]:
# GLUE task configurations
TASK_CONFIG = {
    'rte': {
        'num_labels': 2,
        'metric': 'accuracy',
        'text_fields': ('sentence1', 'sentence2'),
        'epochs': 5,
        'lr': 2e-5,
        'batch_size': 16,
        'max_samples': None
    },
    'mrpc': {
        'num_labels': 2,
        'metric': 'f1',
        'text_fields': ('sentence1', 'sentence2'),
        'epochs': 5,
        'lr': 2e-5,
        'batch_size': 16,
        'max_samples': None
    },
    'cola': {
        'num_labels': 2,
        'metric': 'matthews',
        'text_fields': ('sentence', None),
        'epochs': 5,
        'lr': 2e-5,
        'batch_size': 16,
        'max_samples': None
    },
    'sst2': {
        'num_labels': 2,
        'metric': 'accuracy',
        'text_fields': ('sentence', None),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 8000
    },
    'qnli': {
        'num_labels': 2,
        'metric': 'accuracy',
        'text_fields': ('question', 'sentence'),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 8000
    },
    'qqp': {
        'num_labels': 2,
        'metric': 'f1',
        'text_fields': ('question1', 'question2'),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 12000
    },
    'mnli': {
        'num_labels': 3,
        'metric': 'accuracy',
        'text_fields': ('premise', 'hypothesis'),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 12000,
        'val_split': 'validation_matched'
    }
}

# Sparsity levels for merging (k = keep top k%)
SPARSITY_LEVELS = [5, 10, 20, 40, 60, 80]

# For merging, we'll use binary classification tasks only (exclude MNLI for classifier compatibility)
# Or we can use a shared classifier approach
MERGE_TASKS = ['rte', 'mrpc', 'cola', 'sst2', 'qnli', 'qqp']  # All binary tasks

print(f"Tasks to train and merge: {MERGE_TASKS}")
print(f"Sparsity levels (k): {SPARSITY_LEVELS}%")

## Data Loading

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def load_glue_task(task_name):
    """Load and preprocess a GLUE task."""
    config = TASK_CONFIG[task_name]
    dataset = load_dataset('glue', task_name)
    
    text1_field, text2_field = config['text_fields']
    
    def tokenize_fn(examples):
        if text2_field:
            return tokenizer(
                examples[text1_field],
                examples[text2_field],
                padding='max_length',
                truncation=True,
                max_length=128
            )
        else:
            return tokenizer(
                examples[text1_field],
                padding='max_length',
                truncation=True,
                max_length=128
            )
    
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset['train'].column_names[:-1])
    tokenized = tokenized.rename_column('label', 'labels')
    tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    train_data = tokenized['train']
    val_split = config.get('val_split', 'validation')
    val_data = tokenized[val_split]
    
    max_samples = config['max_samples']
    if max_samples and len(train_data) > max_samples:
        train_data = train_data.shuffle(seed=42).select(range(max_samples))
    
    return train_data, val_data

def compute_metric(preds, labels, metric_name):
    if metric_name == 'accuracy':
        return accuracy_score(labels, preds)
    elif metric_name == 'f1':
        return f1_score(labels, preds, average='binary')
    elif metric_name == 'matthews':
        return matthews_corrcoef(labels, preds)
    else:
        raise ValueError(f"Unknown metric: {metric_name}")

print("✓ Data loading functions defined")

## Train Task Experts with Curvature Tracking

In [ ]:
def train_expert(task_name, base_state_dict):
    """Train a task-specific expert from the base model."""
    config = TASK_CONFIG[task_name]
    print(f"\n{'='*60}")
    print(f"Training {task_name.upper()} expert")
    print(f"{'='*60}")
    
    # Load data
    train_data, val_data = load_glue_task(task_name)
    print(f"Train: {len(train_data)}, Val: {len(val_data)}")
    
    train_loader = DataLoader(train_data, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_data, batch_size=config['batch_size'])
    
    # Initialize model from base
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=config['num_labels']
    ).to(device)
    
    # Load base weights (only encoder, not classifier)
    model_dict = model.state_dict()
    pretrained_dict = {k: v for k, v in base_state_dict.items() if k in model_dict and 'classifier' not in k}
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)
    
    # Store initial weights (w0)
    w0_dict = {}
    curvature_dict = {}
    
    for name, param in model.named_parameters():
        if param.requires_grad and param.dim() == 2 and 'classifier' not in name:
            w0_dict[name] = param.data.clone().cpu()
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device='cpu'),
                'C': torch.zeros(n, device='cpu')
            }
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.01)
    total_steps = len(train_loader) * config['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    
    # Training
    beta2 = 0.999
    
    for epoch in range(config['epochs']):
        model.train()
        total_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            
            # Update curvature (EMA of squared gradients)
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        g2 = param.grad ** 2
                        R_new = g2.mean(dim=1).cpu()
                        C_new = g2.mean(dim=0).cpu()
                        curvature_dict[name]['R'] = beta2 * curvature_dict[name]['R'] + (1 - beta2) * R_new
                        curvature_dict[name]['C'] = beta2 * curvature_dict[name]['C'] + (1 - beta2) * C_new
            
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        print(f"  Epoch {epoch+1} avg loss: {total_loss/len(train_loader):.4f}")
    
    # Compute saliency scores
    saliency_dict = {}
    for name in curvature_dict.keys():
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name].to(device)
        param_change = (param.data - w0) ** 2
        
        R = curvature_dict[name]['R'].to(device).clamp(min=1e-8)
        C = curvature_dict[name]['C'].to(device).clamp(min=1e-8)
        curvature_approx = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        
        saliency_dict[name] = (param_change * curvature_approx).cpu()
    
    # Evaluate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].cpu().numpy())
    
    score = compute_metric(all_preds, all_labels, config['metric'])
    print(f"✓ Expert {config['metric']}: {score:.4f}")
    
    # Extract task vector (delta from base)
    task_vector = {}
    for name in w0_dict.keys():
        param = dict(model.named_parameters())[name]
        task_vector[name] = (param.data - w0_dict[name].to(device)).cpu()
    
    return {
        'task_vector': task_vector,
        'saliency': saliency_dict,
        'curvature': curvature_dict,
        'score': score,
        'val_loader': val_loader,
        'config': config
    }

print("✓ Expert training function defined")

## Train All Experts

In [ ]:
# Get base model state dict
base_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
base_state_dict = {k: v.clone() for k, v in base_model.state_dict().items()}

# Store w0 for merging
w0_global = {}
for name, param in base_model.named_parameters():
    if param.dim() == 2 and 'classifier' not in name:
        w0_global[name] = param.data.clone().cpu()

del base_model
gc.collect()
torch.cuda.empty_cache()

print(f"Base model loaded with {len(w0_global)} weight matrices")

In [ ]:
# Train all experts
experts = {}

for task_name in MERGE_TASKS:
    experts[task_name] = train_expert(task_name, base_state_dict)
    
    # Clean up GPU memory
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("All experts trained!")
print("="*60)
print("\nExpert scores:")
for task_name, expert in experts.items():
    print(f"  {task_name.upper()}: {expert['score']:.4f} ({expert['config']['metric']})")

## Merging Methods Implementation

In [ ]:
def merge_umtam(experts, w0_global, k_percent):
    """
    UMTAM Curvature-Aware Merging (Algorithm 2)
    1. Compute curvature-weighted saliency for each task
    2. Keep top-k% most salient parameters per task
    3. Sign election weighted by curvature
    4. Curvature-weighted averaging
    """
    merged = {}
    
    for name in w0_global.keys():
        task_vectors = []
        saliencies = []
        curvatures = []
        
        for task_name, expert in experts.items():
            if name in expert['task_vector']:
                task_vectors.append(expert['task_vector'][name])
                saliencies.append(expert['saliency'][name])
                
                # Compute full curvature matrix
                R = expert['curvature'][name]['R'].clamp(min=1e-8)
                C = expert['curvature'][name]['C'].clamp(min=1e-8)
                curv = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
                curvatures.append(curv)
        
        if not task_vectors:
            merged[name] = w0_global[name].clone()
            continue
        
        # Stack tensors
        task_vectors = torch.stack(task_vectors)  # [T, m, n]
        saliencies = torch.stack(saliencies)      # [T, m, n]
        curvatures = torch.stack(curvatures)      # [T, m, n]
        
        # Create masks: top-k% saliency per task
        masks = []
        for t in range(len(experts)):
            sal = saliencies[t].flatten()
            k = int(len(sal) * k_percent / 100)
            k = max(1, min(k, len(sal) - 1))
            threshold = torch.kthvalue(sal, len(sal) - k).values
            mask = (saliencies[t] >= threshold).float()
            masks.append(mask)
        masks = torch.stack(masks)  # [T, m, n]
        
        # Masked task vectors
        masked_vectors = task_vectors * masks
        
        # Sign election: weighted by curvature
        signs = torch.sign(masked_vectors)
        weighted_signs = signs * curvatures * masks
        elected_sign = torch.sign(weighted_signs.sum(dim=0))
        
        # Keep only agreeing parameters
        agreement = (signs == elected_sign.unsqueeze(0)).float() * masks
        
        # Curvature-weighted average
        weights = curvatures * agreement
        weight_sum = weights.sum(dim=0).clamp(min=1e-8)
        
        merged_delta = (masked_vectors * agreement * curvatures).sum(dim=0) / weight_sum
        merged_delta = torch.where(weight_sum > 1e-8, merged_delta, torch.zeros_like(merged_delta))
        
        merged[name] = w0_global[name] + merged_delta
    
    return merged


def merge_ties(experts, w0_global, k_percent):
    """
    TIES-Merging: Trim + Elect Sign + Merge
    Uses magnitude-based trimming (not curvature)
    """
    merged = {}
    
    for name in w0_global.keys():
        task_vectors = []
        
        for task_name, expert in experts.items():
            if name in expert['task_vector']:
                task_vectors.append(expert['task_vector'][name])
        
        if not task_vectors:
            merged[name] = w0_global[name].clone()
            continue
        
        task_vectors = torch.stack(task_vectors)
        
        # Trim: keep top-k% by magnitude
        masks = []
        for t in range(len(task_vectors)):
            mag = task_vectors[t].abs().flatten()
            k = int(len(mag) * k_percent / 100)
            k = max(1, min(k, len(mag) - 1))
            threshold = torch.kthvalue(mag, len(mag) - k).values
            mask = (task_vectors[t].abs() >= threshold).float()
            masks.append(mask)
        masks = torch.stack(masks)
        
        masked_vectors = task_vectors * masks
        
        # Elect sign: majority vote weighted by magnitude
        signs = torch.sign(masked_vectors)
        weighted_signs = signs * masked_vectors.abs()
        elected_sign = torch.sign(weighted_signs.sum(dim=0))
        
        # Keep agreeing
        agreement = (signs == elected_sign.unsqueeze(0)).float() * masks
        
        # Simple average of agreeing
        count = agreement.sum(dim=0).clamp(min=1)
        merged_delta = (masked_vectors * agreement).sum(dim=0) / count
        
        merged[name] = w0_global[name] + merged_delta
    
    return merged


def merge_dare_ties(experts, w0_global, k_percent, drop_rate=0.9):
    """
    DARE + TIES: Random drop then TIES
    """
    merged = {}
    
    for name in w0_global.keys():
        task_vectors = []
        
        for task_name, expert in experts.items():
            if name in expert['task_vector']:
                tv = expert['task_vector'][name].clone()
                # DARE: random drop and rescale
                mask = torch.bernoulli(torch.ones_like(tv) * (1 - drop_rate))
                tv = tv * mask / (1 - drop_rate + 1e-8)
                task_vectors.append(tv)
        
        if not task_vectors:
            merged[name] = w0_global[name].clone()
            continue
        
        task_vectors = torch.stack(task_vectors)
        
        # TIES on dropped vectors
        masks = []
        for t in range(len(task_vectors)):
            mag = task_vectors[t].abs().flatten()
            k = int(len(mag) * k_percent / 100)
            k = max(1, min(k, len(mag) - 1))
            threshold = torch.kthvalue(mag, len(mag) - k).values
            mask = (task_vectors[t].abs() >= threshold).float()
            masks.append(mask)
        masks = torch.stack(masks)
        
        masked_vectors = task_vectors * masks
        signs = torch.sign(masked_vectors)
        weighted_signs = signs * masked_vectors.abs()
        elected_sign = torch.sign(weighted_signs.sum(dim=0))
        agreement = (signs == elected_sign.unsqueeze(0)).float() * masks
        count = agreement.sum(dim=0).clamp(min=1)
        merged_delta = (masked_vectors * agreement).sum(dim=0) / count
        
        merged[name] = w0_global[name] + merged_delta
    
    return merged


def merge_linear(experts, w0_global):
    """
    Linear averaging (simple task arithmetic)
    """
    merged = {}
    n_tasks = len(experts)
    
    for name in w0_global.keys():
        task_vectors = []
        
        for task_name, expert in experts.items():
            if name in expert['task_vector']:
                task_vectors.append(expert['task_vector'][name])
        
        if task_vectors:
            avg_delta = torch.stack(task_vectors).mean(dim=0)
            merged[name] = w0_global[name] + avg_delta
        else:
            merged[name] = w0_global[name].clone()
    
    return merged


def merge_magnitude_weighted(experts, w0_global):
    """
    Magnitude-weighted averaging
    """
    merged = {}
    
    for name in w0_global.keys():
        task_vectors = []
        magnitudes = []
        
        for task_name, expert in experts.items():
            if name in expert['task_vector']:
                tv = expert['task_vector'][name]
                task_vectors.append(tv)
                magnitudes.append(tv.abs())
        
        if task_vectors:
            task_vectors = torch.stack(task_vectors)
            magnitudes = torch.stack(magnitudes)
            
            weights = magnitudes / magnitudes.sum(dim=0).clamp(min=1e-8)
            weighted_delta = (task_vectors * weights).sum(dim=0)
            merged[name] = w0_global[name] + weighted_delta
        else:
            merged[name] = w0_global[name].clone()
    
    return merged

print("✓ All merging methods defined")

## Evaluation Function

In [ ]:
def evaluate_merged_model(merged_weights, experts, w0_global):
    """
    Evaluate merged model on all tasks.
    Returns dict of {task: score}
    """
    results = {}
    
    for task_name, expert in experts.items():
        config = expert['config']
        val_loader = expert['val_loader']
        
        # Create model and load merged weights
        model = BertForSequenceClassification.from_pretrained(
            'bert-base-uncased',
            num_labels=config['num_labels']
        ).to(device)
        
        # Load merged weights
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in merged_weights:
                    param.data.copy_(merged_weights[name].to(device))
        
        # Evaluate
        model.eval()
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                preds = outputs.logits.argmax(dim=-1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(batch['labels'].cpu().numpy())
        
        score = compute_metric(all_preds, all_labels, config['metric'])
        results[task_name] = score
        
        del model
        torch.cuda.empty_cache()
    
    return results

print("✓ Evaluation function defined")

## Run Merging Experiments

In [ ]:
# Run all merging experiments
all_results = {}

# Methods that don't depend on k (sparsity)
print("\n" + "="*60)
print("Evaluating Linear Averaging...")
print("="*60)
merged = merge_linear(experts, w0_global)
scores = evaluate_merged_model(merged, experts, w0_global)
all_results['Linear'] = {k: {'all': scores} for k in SPARSITY_LEVELS}
print(f"Average: {np.mean(list(scores.values())):.4f}")
linear_scores = scores

print("\n" + "="*60)
print("Evaluating Magnitude-Weighted...")
print("="*60)
merged = merge_magnitude_weighted(experts, w0_global)
scores = evaluate_merged_model(merged, experts, w0_global)
all_results['MagWeight'] = {k: {'all': scores} for k in SPARSITY_LEVELS}
print(f"Average: {np.mean(list(scores.values())):.4f}")

# Methods that depend on k
for k in SPARSITY_LEVELS:
    print(f"\n" + "="*60)
    print(f"Sparsity k = {k}%")
    print("="*60)
    
    # UMTAM
    print("  UMTAM...")
    merged = merge_umtam(experts, w0_global, k)
    scores = evaluate_merged_model(merged, experts, w0_global)
    if 'UMTAM' not in all_results:
        all_results['UMTAM'] = {}
    all_results['UMTAM'][k] = scores
    print(f"    Average: {np.mean(list(scores.values())):.4f}")
    
    # TIES
    print("  TIES...")
    merged = merge_ties(experts, w0_global, k)
    scores = evaluate_merged_model(merged, experts, w0_global)
    if 'TIES' not in all_results:
        all_results['TIES'] = {}
    all_results['TIES'][k] = scores
    print(f"    Average: {np.mean(list(scores.values())):.4f}")
    
    # DARE+TIES
    print("  DARE+TIES...")
    merged = merge_dare_ties(experts, w0_global, k)
    scores = evaluate_merged_model(merged, experts, w0_global)
    if 'DARE+TIES' not in all_results:
        all_results['DARE+TIES'] = {}
    all_results['DARE+TIES'][k] = scores
    print(f"    Average: {np.mean(list(scores.values())):.4f}")

print("\n" + "="*60)
print("All merging experiments completed!")
print("="*60)

## Results Summary

In [ ]:
import pandas as pd

# Create summary table
print("\n" + "="*80)
print("MERGING RESULTS SUMMARY")
print("="*80)

# Expert baselines
print("\nExpert Baselines (single-task fine-tuned):")
for task_name, expert in experts.items():
    print(f"  {task_name.upper()}: {expert['score']:.4f}")

# Best results per method
print("\n" + "-"*80)
print("Best Average Score per Method:")
print("-"*80)

method_best = {}
for method in ['UMTAM', 'TIES', 'DARE+TIES', 'Linear', 'MagWeight']:
    if method in ['Linear', 'MagWeight']:
        scores = all_results[method][SPARSITY_LEVELS[0]]['all']
        avg = np.mean(list(scores.values()))
        method_best[method] = {'avg': avg, 'k': 'N/A', 'scores': scores}
    else:
        best_avg = 0
        best_k = 0
        best_scores = None
        for k in SPARSITY_LEVELS:
            scores = all_results[method][k]
            avg = np.mean(list(scores.values()))
            if avg > best_avg:
                best_avg = avg
                best_k = k
                best_scores = scores
        method_best[method] = {'avg': best_avg, 'k': best_k, 'scores': best_scores}

# Sort by average
sorted_methods = sorted(method_best.items(), key=lambda x: x[1]['avg'], reverse=True)

for method, data in sorted_methods:
    print(f"  {method:12s}: {data['avg']:.4f} (k={data['k']})")
    for task, score in data['scores'].items():
        print(f"               {task.upper()}: {score:.4f}")

## Detailed Results by Sparsity

In [ ]:
# Create detailed table
print("\n" + "="*80)
print("DETAILED RESULTS BY SPARSITY LEVEL (k)")
print("="*80)

for k in SPARSITY_LEVELS:
    print(f"\nk = {k}%")
    print("-" * 70)
    
    # Header
    tasks = list(experts.keys())
    header = f"{'Method':12s}" + "".join([f"{t.upper():>10s}" for t in tasks]) + f"{'Average':>10s}"
    print(header)
    print("-" * 70)
    
    for method in ['UMTAM', 'TIES', 'DARE+TIES', 'Linear', 'MagWeight']:
        if method in ['Linear', 'MagWeight']:
            scores = all_results[method][k]['all']
        else:
            scores = all_results[method][k]
        
        row = f"{method:12s}"
        for task in tasks:
            row += f"{scores[task]:>10.4f}"
        avg = np.mean(list(scores.values()))
        row += f"{avg:>10.4f}"
        print(row)

## Generate LaTeX Tables

In [ ]:
# Main results table
tasks = list(experts.keys())
n_tasks = len(tasks)

latex = r"""
\begin{table*}[t]
\centering
\caption{Multi-task merging results on """ + str(n_tasks) + r""" GLUE tasks. Best result per column in bold. UMTAM uses curvature-aware saliency; baselines use magnitude-based or random selection.}
\label{tab:merging_6tasks}
\begin{tabular}{l|""" + "c" * n_tasks + r"|c}
\toprule
\textbf{Method} & """ + " & ".join([f"\\textbf{{{t.upper()}}}" for t in tasks]) + r" & \textbf{Average} \\
\midrule
"""

# Find best k for each method
for method in ['UMTAM', 'TIES', 'DARE+TIES', 'Linear', 'MagWeight']:
    data = method_best[method]
    scores = data['scores']
    avg = data['avg']
    
    # Check if this method wins each column
    row_vals = []
    for task in tasks:
        val = scores[task]
        # Check if best
        is_best = True
        for m2 in method_best:
            if method_best[m2]['scores'][task] > val:
                is_best = False
                break
        if is_best:
            row_vals.append(f"\\textbf{{{val:.3f}}}")
        else:
            row_vals.append(f"{val:.3f}")
    
    # Check avg
    is_best_avg = all(avg >= method_best[m2]['avg'] for m2 in method_best)
    avg_str = f"\\textbf{{{avg:.3f}}}" if is_best_avg else f"{avg:.3f}"
    
    latex += f"{method} & " + " & ".join(row_vals) + f" & {avg_str} \\\\\n"

latex += r"""
\bottomrule
\end{tabular}
\end{table*}
"""

print(latex)

In [ ]:
# Sparsity ablation table
latex2 = r"""
\begin{table}[t]
\centering
\caption{Effect of sparsity level $k$ on merging performance (average across """ + str(n_tasks) + r""" tasks).}
\label{tab:sparsity_ablation_6tasks}
\begin{tabular}{l|cccccc}
\toprule
\textbf{Method} & \textbf{k=5\%} & \textbf{k=10\%} & \textbf{k=20\%} & \textbf{k=40\%} & \textbf{k=60\%} & \textbf{k=80\%} \\
\midrule
"""

for method in ['UMTAM', 'TIES', 'DARE+TIES']:
    row = f"{method}"
    avgs = []
    for k in SPARSITY_LEVELS:
        scores = all_results[method][k]
        avg = np.mean(list(scores.values()))
        avgs.append(avg)
    
    best_idx = np.argmax(avgs)
    for i, avg in enumerate(avgs):
        if i == best_idx:
            row += f" & \\textbf{{{avg:.3f}}}"
        else:
            row += f" & {avg:.3f}"
    row += " \\\\\n"
    latex2 += row

latex2 += r"""
\bottomrule
\end{tabular}
\end{table}
"""

print(latex2)

## Visualization

In [ ]:
# Plot sparsity vs performance
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

colors = {'UMTAM': 'blue', 'TIES': 'red', 'DARE+TIES': 'green'}
markers = {'UMTAM': 'o', 'TIES': 's', 'DARE+TIES': '^'}

for method in ['UMTAM', 'TIES', 'DARE+TIES']:
    avgs = []
    for k in SPARSITY_LEVELS:
        scores = all_results[method][k]
        avgs.append(np.mean(list(scores.values())))
    
    ax.plot(SPARSITY_LEVELS, avgs, 
            color=colors[method], 
            marker=markers[method],
            label=method,
            linewidth=2,
            markersize=8)

# Add horizontal lines for non-sparse methods
linear_avg = np.mean(list(all_results['Linear'][SPARSITY_LEVELS[0]]['all'].values()))
ax.axhline(y=linear_avg, color='orange', linestyle='--', label='Linear', linewidth=2)

magweight_avg = np.mean(list(all_results['MagWeight'][SPARSITY_LEVELS[0]]['all'].values()))
ax.axhline(y=magweight_avg, color='purple', linestyle=':', label='MagWeight', linewidth=2)

ax.set_xlabel('Sparsity k (%)', fontsize=12)
ax.set_ylabel('Average Score', fontsize=12)
ax.set_title(f'Multi-Task Merging: {n_tasks} GLUE Tasks', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 85])

plt.tight_layout()
plt.savefig('merging_6tasks_sparsity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved as 'merging_6tasks_sparsity.png'")

In [ ]:
# Bar chart of best results
fig, ax = plt.subplots(figsize=(12, 6))

methods = ['UMTAM', 'TIES', 'DARE+TIES', 'Linear', 'MagWeight']
tasks = list(experts.keys())
x = np.arange(len(tasks))
width = 0.15

colors = ['blue', 'red', 'green', 'orange', 'purple']

for i, method in enumerate(methods):
    scores = [method_best[method]['scores'][t] for t in tasks]
    ax.bar(x + i * width, scores, width, label=method, color=colors[i], alpha=0.8)

# Add expert baseline
expert_scores = [experts[t]['score'] for t in tasks]
ax.scatter(x + 2 * width, expert_scores, color='black', marker='*', s=100, zorder=5, label='Expert')

ax.set_xlabel('Task', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Multi-Task Merging Results ({n_tasks} GLUE Tasks)', fontsize=14, fontweight='bold')
ax.set_xticks(x + 2 * width)
ax.set_xticklabels([t.upper() for t in tasks])
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('merging_6tasks_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved as 'merging_6tasks_comparison.png'")

## Save Results

In [ ]:
import json

# Prepare results for JSON serialization
save_results = {
    'expert_scores': {t: experts[t]['score'] for t in experts},
    'merging_results': {},
    'method_best': {m: {'avg': d['avg'], 'k': d['k'], 'scores': d['scores']} 
                   for m, d in method_best.items()}
}

for method in all_results:
    save_results['merging_results'][method] = {}
    for k in all_results[method]:
        if method in ['Linear', 'MagWeight']:
            save_results['merging_results'][method][k] = all_results[method][k]['all']
        else:
            save_results['merging_results'][method][k] = all_results[method][k]

with open('merging_6tasks_results.json', 'w') as f:
    json.dump(save_results, f, indent=2)

print("✓ Results saved to:")
print("  - merging_6tasks_results.json")
print("  - merging_6tasks_sparsity.png")
print("  - merging_6tasks_comparison.png")

## Summary Statistics

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

# UMTAM improvements over baselines
umtam_avg = method_best['UMTAM']['avg']

print(f"\nUMTAM average: {umtam_avg:.4f}")
print("\nImprovement over baselines:")

for method in ['TIES', 'DARE+TIES', 'Linear', 'MagWeight']:
    baseline_avg = method_best[method]['avg']
    improvement = ((umtam_avg - baseline_avg) / baseline_avg) * 100
    print(f"  vs {method:12s}: {improvement:+.2f}%")

# Win rate
print("\nTask-level wins (UMTAM vs others):")
umtam_scores = method_best['UMTAM']['scores']
for method in ['TIES', 'DARE+TIES', 'Linear', 'MagWeight']:
    other_scores = method_best[method]['scores']
    wins = sum(1 for t in tasks if umtam_scores[t] > other_scores[t])
    print(f"  vs {method:12s}: {wins}/{len(tasks)} tasks")